# 비유의성의 추정 정밀도와 식별변동 진단

종로구·마포구 / 외식업 10종 / 2023~2025년 12분기 / 기존 최종 결과를 고정한 오프라인 진단

**질문:** 현재 추정값의 불확실성은 얼마나 큰가? 고정효과를 제거한 뒤 실제로 남는 공연 노출 변동은 얼마나 되며, 어떤 상권에 집중되는가?

기존 05·11의 주모형과 표본만 재현한다. 유의성을 얻기 위한 반경·표본·변환·모형 탐색을 하지 않는다. 기존 입력과 결과를 수정하지 않으며, 출력은 `reports/20261006/null_diagnostics/inference_*`에만 저장한다. 관측된 계수로 사후검정력을 계산하지 않는다.

**해석 원칙:** 비유의성은 효과 부재의 증거가 아니며, 모든 비유의성이 표본 부족 때문인 것도 아니다. 계산으로 확인되는 정밀도·변동·집중도와 측정오차·교란에 관한 가설을 구별한다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import os
for key in ['OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','OMP_NUM_THREADS']:
    os.environ[key]='1'
import ast, json, hashlib, platform
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy import linalg, stats
from statsmodels.stats.multitest import multipletests
from IPython.display import display, Markdown
ROOT=PROJECT_ROOT
OUT=project_path('reports/20261006/null_diagnostics')
OUT.mkdir(parents=True,exist_ok=True)
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
read_json=lambda p:project_json_loads(p.read_text(encoding='utf-8'))
R=project_path('reports/20261006'); TW=R/'time_windows'
nb05=project_path('jupyter-notebook/20261006/notebooks/05_외식업_고정효과분석.ipynb')
nb11=project_path('jupyter-notebook/20261006/notebooks/11_공연시간창_매출분석.ipynb')
s05=read_json(R/'09_analysis_summary.json'); s11=read_json(TW/'11_analysis_summary.json')
protected={s05['input_file']:s05['input_sha256'],**s05['output_sha256'],**s11['input_sha256']}
protected.update({(TW/name).relative_to(ROOT).as_posix():value for name,value in s11['output_sha256'].items()})
for p in [nb05,nb11,R/'09_analysis_summary.json',TW/'11_analysis_summary.json']:
    protected[p.relative_to(ROOT).as_posix()]=sha(p)
assert all(sha(project_path(path))==value for path,value in protected.items())
pd.set_option('display.max_rows',45)
pd.set_option('display.float_format',lambda x:f'{x:,.6f}')
print(f'고정 입력·결과·소스 {len(protected)}개 SHA-256 확인')

## 1. 실제로 추정하는 효과

05의 업종별 모형은 `log(1+분기 총매출)` 또는 야간 매출 비중(%p)을 `log(1+500m 등록 공연 ID 수)`에 회귀한다. 업종별 상권 고정효과와 분기 고정효과를 제거하므로 공연이 많은 지역과 적은 지역의 단순 매출 차이가 아니다. 같은 상권의 분기 변화 중 업종 전체의 공통 분기 변화로 설명되지 않는 관계를 추정한다. 등록 ID는 실제 회차·관객 수가 아니다.

11은 `log(1+6개 원시간구간 매출)`을 `log(1+예정 시간창 노출)`에 회귀한다. 상권×업종×시간대, 업종×분기×시간대, 상권×업종×분기 고정효과를 모두 제거한다. 같은 분기의 전반적 매출 수준은 흡수되므로 **시간대별 상대 매출 패턴의 변화**를 추정하며, 공연 후 실제 3시간 매출이나 총매출 증가 효과를 추정하지 않는다. 0~6시간과 0~3시간·3~6시간은 중첩된 별도 모형이다.

군집은 두 모형 모두 상권이며, 명목 행 수가 독립 표본 수는 아니다. 현재 추정 방식은 인접 상권 사이의 잔차 공간 상관을 해결하지 않는다.

## 2. 기존 함수만 읽어 동일 표본 재현

기존 노트북의 지정 함수 정의만 AST로 읽는다. 저장·그림·전체 실행 셀은 실행하지 않는다. 이렇게 원래 표본 필터와 흡수 계산을 유지하되 기존 파일은 수정하지 않는다. 독립적으로 저장 CSV의 계수·표준오차·신뢰구간·p값을 대조한다.

In [ ]:
def load_functions(path,names,namespace):
    found={}
    for cell in read_json(path)['cells']:
        if cell['cell_type']!='code': continue
        source=''.join(cell['source'])
        if any(line.lstrip().startswith(('%','!')) for line in source.splitlines()): continue
        tree=ast.parse(source)
        for node in tree.body:
            if isinstance(node,ast.FunctionDef) and node.name in names:
                found[node.name]=node
    assert set(found)==set(names),(path,set(names)-set(found))
    module=ast.Module(body=[found[name] for name in names],type_ignores=[])
    exec(compile(ast.fix_missing_locations(module),str(path),'exec'),namespace)
    return namespace

KEY5=['area_id','industry_id','quarter']
p5=pd.read_csv(project_path(s05['input_file']),dtype={k:str for k in KEY5})
p5['entity']=p5.area_id+'_'+p5.industry_id
p5['log_sales']=np.log1p(p5.total_sales.where(p5.total_sales.ge(0)))
p5['night_share_pct']=(100*p5.night_sales/p5.total_sales).where(p5.total_sales.gt(0)&p5.night_sales.ge(0)&p5.night_sales.le(p5.total_sales))
p5['log_stores']=np.log1p(p5.regular_store_count.where(p5.regular_store_count.ge(0)))
p5['log_total_stores']=np.log1p(p5.total_store_count.where(p5.total_store_count.ge(0)))
INDUSTRIES=dict(p5[['industry_id','industry']].drop_duplicates().sort_values('industry_id').to_numpy())
ns5={'np':np,'pd':pd,'linalg':linalg,'panel':p5,'KEY':KEY5,'INDUSTRIES':INDUSTRIES}
load_functions(nb05,['prepare_frame','absorb_effects','fit_food_fe'],ns5)
models5={o:ns5['fit_food_fe'](ns5['prepare_frame'](o),o) for o in ['log_sales','night_share_pct']}

BINS=['00_06','06_11','11_14','14_17','17_21','21_24']
WINDOWS={'post03':'종료 후 0~3시간','post36':'종료 후 3~6시간','post06':'종료 후 0~6시간','pre03':'시작 전 3시간'}
D=project_path('dataset/analysis/final/time_windows')
sales=pd.read_csv(D/'sales_time_panel.csv',dtype={k:str for k in KEY5+['time_bin']})
expo=pd.read_csv(D/'area_quarter_timebin_exposure.csv',dtype={k:str for k in ['area_id','quarter','time_bin']})
xcols=['known_exposure_'+w for w in WINDOWS]
p11=sales.merge(expo[['area_id','quarter','time_bin','availability_pct']+xcols],on=['area_id','quarter','time_bin'],validate='many_to_one')
p11['y']=np.log1p(p11.sales_amount.where(p11.sales_amount.ge(0)))
ns11={'np':np,'pd':pd,'linalg':linalg,'stats':stats,'panel':p11,'BINS':BINS,'BASE_KEY':KEY5,'INDUSTRY_IDS':list(INDUSTRIES)}
load_functions(nb11,['prepare','absorb_area_quarter','absorb_time_effects','model_blocks','fit_system','coefficient_row'],ns11)
models11={}
for window in WINDOWS:
    blocks,audit=ns11['model_blocks'](ns11['prepare'](window,90),True)
    models11[window]=(blocks,ns11['fit_system'](blocks,True),ns11['fit_system'](blocks,False))
print('기존 05 주모형 2개와 11 주표본 4창 재현 완료')

In [ ]:
ref5=pd.read_csv(R/'09_industry_effects.csv').query("scenario == 'main_500'").copy()
ref11=pd.read_csv(TW/'11_industry_effects.csv')
refcommon=pd.read_csv(TW/'11_common_effects.csv')
checks=[]; effect_rows=[]
checkcols=['coefficient','standard_error','ci_low','ci_high','p_value']
def infer(m,pos):
    beta=float(m['beta'][pos]); se=float(np.sqrt(m['cov'][pos,pos])); crit=stats.t.ppf(.975,m['df'])
    return dict(coefficient=beta,standard_error=se,ci_low=beta-crit*se,ci_high=beta+crit*se,
                p_value=float(2*stats.t.sf(abs(beta/se),m['df'])),df=m['df'],nobs_total=m['nobs'],
                clusters_total=m['clusters'],rank=m['rank'],correction=m['correction'])
member=pd.read_csv(R/'09_model_sample_membership.csv',dtype={k:str for k in KEY5})
for outcome,m in models5.items():
    original=member.loc[member.scenario.eq('main_500')&member.outcome.eq(outcome),KEY5]
    reproduced=pd.concat([b['frame'][KEY5] for b in m['blocks']],ignore_index=True)
    assert set(map(tuple,original.to_numpy()))==set(map(tuple,reproduced.to_numpy()))
    for b in m['blocks']:
        row=dict(family='primary20',outcome=outcome,window='none',industry_id=b['industry_id'],industry=INDUSTRIES[b['industry_id']],
                 nobs_industry=len(b['frame']),clusters_industry=b['frame'].area_id.nunique(),**infer(m,b['offset']))
        ref=ref5.loc[ref5.outcome.eq(outcome)&ref5.industry_id.eq(b['industry_id'])].iloc[0]
        error=max(abs(row[k]-ref[k]) for k in checkcols)
        assert error<1e-10,(outcome,b['industry_id'],error)
        checks.append(dict(family=row['family'],outcome=outcome,window='none',industry_id=b['industry_id'],max_absolute_error=error,passed=True))
        effect_rows.append(row)
for window,(blocks,common,heterogeneous) in models11.items():
    for pos,b in enumerate(blocks):
        row=dict(family='time40',outcome='log_timebin_sales',window=window,industry_id=b['industry_id'],industry=INDUSTRIES[b['industry_id']],
                 nobs_industry=len(b['frame']),clusters_industry=b['frame'].area_id.nunique(),**infer(heterogeneous,pos))
        ref=ref11.loc[ref11.window.eq(window)&ref11.industry_id.eq(b['industry_id'])].iloc[0]
        error=max(abs(row[k]-ref[k]) for k in checkcols)
        assert error<1e-10 and row['nobs_industry']==ref.industry_rows and row['clusters_industry']==ref.industry_clusters
        checks.append(dict(family=row['family'],outcome=row['outcome'],window=window,industry_id=b['industry_id'],max_absolute_error=error,passed=True))
        effect_rows.append(row)
    row=dict(family='time_common4',outcome='log_timebin_sales',window=window,industry_id='COMMON',industry='공통 기울기',
             nobs_industry=common['nobs'],clusters_industry=common['clusters'],**infer(common,0))
    ref=refcommon.loc[refcommon.window.eq(window)].iloc[0]
    error=max(abs(row[k]-ref[k]) for k in checkcols)
    assert error<1e-10
    checks.append(dict(family=row['family'],outcome=row['outcome'],window=window,industry_id='COMMON',max_absolute_error=error,passed=True))
    effect_rows.append(row)
effects=pd.DataFrame(effect_rows)
effects['p_adjusted']=np.nan
for family,method in [('primary20','fdr_bh'),('time40','fdr_bh'),('time_common4','holm')]:
    ix=effects.family.eq(family)
    effects.loc[ix,'p_adjusted']=multipletests(effects.loc[ix,'p_value'],method=method)[1]
    effects.loc[ix,'adjustment']=method
for family,ref,keycols,pcol in [('primary20',ref5,['outcome','industry_id'],'p_fdr_primary'),('time40',ref11,['window','industry_id'],'p_bh_40'),('time_common4',refcommon,['window'],'p_holm_4')]:
    cmp=effects.loc[effects.family.eq(family)].merge(ref[keycols+[pcol]],on=keycols,validate='one_to_one')
    assert np.max(np.abs(cmp.p_adjusted-cmp[pcol]))<1e-10
reproduction=pd.DataFrame(checks)
display(reproduction.groupby('family').agg(검증수=('passed','size'),최대오차=('max_absolute_error','max')))
display(effects.groupby('family').agg(검정수=('p_value','size'),보정전유의=('p_value',lambda x:int(x.lt(.05).sum())),보정후유의=('p_adjusted',lambda x:int(x.lt(.05).sum()))))

**재현 해석:** 계수와 군집 추론이 저장 결과와 같다는 것은 계산 재현성의 확인이다. 공연 노출의 정확성이나 인과적 식별을 검증했다는 뜻은 아니다. 개별 95% 신뢰구간과 BH/Holm 보정 p값은 서로 다른 결과이며, 신뢰구간은 다중비교 동시 구간이 아니다.

## 3. 고정효과 제거 전후의 노출 변동과 정보 집중

원시 변동은 분석에 투입된 `x=log(1+노출)`의 전체 평균 주위 제곱합이다. 05에서는 상권 평균 제거, 11에서는 상권×시간대 평균 제거 후의 변동도 보고한다. 최종 잔차 제곱합/원시 제곱합은 모형이 기울기 추정에 사용하는 변동 비중이다. FE가 필요 없는 변동을 없앴다는 뜻이 아니며, FE를 빼면 더 좋은 모형이 된다는 근거로 사용하지 않는다.

상권별 정보 비중은 `sum(잔차 x²)`의 비중이다. 상위 5개 비중과 `1/sum(비중²)`를 제시한다. 후자는 같은 정보량을 나눠 가진 상권의 개수에 해당하는 **기술적 정보 분산 지표**이며 실제 독립 군집 수·검정 자유도·유효 표본 크기가 아니다. 원시 노출 변화가 전혀 없는 상권도 공통 분기 변화에 대한 비교 역할로 잔차 x가 0이 아닐 수 있다.

In [ ]:
diagnostics=[]; cluster_rows=[]
def diagnose(family,outcome,window,industry_id,part,xraw,xres,yres,beta,correction,se):
    d=part.copy().reset_index(drop=True)
    d['_xraw']=np.asarray(xraw); d['_xres']=np.asarray(xres); d['_yres']=np.asarray(yres)
    within_groups=['area_id','time_bin'] if 'time_bin' in d else ['area_id']
    raw=d._xraw.to_numpy(); xr=d._xres.to_numpy(); yr=d._yres.to_numpy()
    raw_sst=float(np.sum((raw-raw.mean())**2)); residual_ss=float(xr@xr)
    within=d._xraw-d.groupby(within_groups)._xraw.transform('mean')
    ranges=d.groupby(within_groups)._xraw.agg(['min','max'])
    changing=(ranges['max']-ranges['min']).gt(1e-10)
    if isinstance(changing.index,pd.MultiIndex): changing=changing.groupby(level='area_id').any()
    spans=(ranges['max']-ranges['min']).ge(np.log(2))
    if isinstance(spans.index,pd.MultiIndex): spans=spans.groupby(level='area_id').any()
    d['_quarter_index']=d.quarter.str[:4].astype(int)*4+d.quarter.str[-1].astype(int)
    adjacent=d.sort_values(within_groups+['_quarter_index'])
    groups=adjacent.groupby(within_groups,sort=False)
    dq=groups._quarter_index.diff(); dx=groups._xraw.diff()
    actual=dx.loc[dq.eq(1)].abs(); nonzero=actual.loc[actual.gt(1e-10)]
    info=d.assign(_info=xr*xr).groupby('area_id')._info.sum()
    shares=(info/info.sum()).sort_values(ascending=False)
    score=d.assign(_score=xr*(yr-xr*beta)).groupby('area_id')._score.sum()
    meat=score.pow(2)
    recreated_se=float(np.sqrt(correction*meat.sum())/residual_ss)
    assert abs(recreated_se-se)<1e-10,(family,window,industry_id,recreated_se,se)
    r2=float((xr@yr)**2/(residual_ss*(yr@yr))) if yr@yr>0 else np.nan
    record=dict(family=family,outcome=outcome,window=window,industry_id=industry_id,industry=INDUSTRIES[industry_id],
        observations=len(d),clusters=d.area_id.nunique(),quarters=d.quarter.nunique(),
        raw_log_exposure_sd=float(raw.std()),area_or_area_bin_demeaned_x_sd=float(within.std(ddof=0)),
        residual_x_sd=float(xr.std()),raw_centered_x_ss=raw_sst,area_or_area_bin_demeaned_x_ss=float(within@within),
        residual_x_ss=residual_ss,fe_variance_retained_pct=100*residual_ss/raw_sst,
        fe_variance_removed_pct=100*(1-residual_ss/raw_sst),
        raw_temporally_changing_clusters=int(changing.sum()),raw_no_change_clusters=int((~changing).sum()),
        nonzero_residual_information_clusters=int(info.gt(residual_ss*1e-12).sum()),
        clusters_spanning_log2_same_bin=int(spans.sum()),
        top1_information_share=float(shares.iloc[0]),top5_information_share=float(shares.head(5).sum()),
        inverse_herfindahl_information=float(1/shares.pow(2).sum()),
        adjacent_quarter_pairs=int(len(actual)),nonzero_adjacent_quarter_pairs=int(len(nonzero)),
        median_nonzero_adjacent_abs_delta_logx=float(nonzero.median()) if len(nonzero) else np.nan,
        adjacent_quarter_pairs_ge_log2=int(actual.ge(np.log(2)).sum()),
        partial_within_r2=r2,residual_y_sd=float(yr.std()),recreated_cluster_se=recreated_se)
    if family=='time40':
        z2,_=ns11['absorb_time_effects'](d,['x'],False)
        record['twoway_residual_x_ss']=float(z2[:,0]@z2[:,0])
        record['threeway_vs_twoway_retained_pct']=100*residual_ss/record['twoway_residual_x_ss']
    diagnostics.append(record)
    for rank,(area,share) in enumerate(shares.items(),1):
        cluster_rows.append(dict(family=family,outcome=outcome,window=window,industry_id=industry_id,
          area_id=area,information_rank=rank,residual_x_ss=float(info.loc[area]),information_share=float(share),
          score_squared=float(meat.loc[area]),score_variance_share=float(meat.loc[area]/meat.sum())))
for outcome,m in models5.items():
    for b in m['blocks']:
        diagnose('primary20',outcome,'none',b['industry_id'],b['frame'],b['frame'].log_activity,b['x'][:,0],b['y'],
                 b['beta'][0],m['correction'],np.sqrt(m['cov'][b['offset'],b['offset']]))
for window,(blocks,common,m) in models11.items():
    for i,b in enumerate(blocks):
        diagnose('time40','log_timebin_sales',window,b['industry_id'],b['frame'],b['frame'].x,b['x'],b['y'],
                 m['beta'][i],m['correction'],np.sqrt(m['cov'][i,i]))
diagnostics=pd.DataFrame(diagnostics); cluster_info=pd.DataFrame(cluster_rows)
# 기존 정보집중 표와도 대조한다.
old=pd.read_csv(R/'09_information_concentration.csv')
a=diagnostics.query("family == 'primary20'").merge(old,on=['outcome','industry_id'],suffixes=('_new','_old'),validate='one_to_one')
for name in ['top5_information_share','inverse_herfindahl_information']:
    assert np.max(np.abs(a[name+'_new']-a[name+'_old']))<1e-10
old11=pd.read_csv(TW/'11_information_concentration.csv')
a=diagnostics.query("family == 'time40'").merge(old11,on=['window','industry_id'],suffixes=('_new','_old'),validate='one_to_one')
assert np.max(np.abs(a.top5_information_share_new-a.top5_information_share_old))<1e-10
focus=diagnostics.loc[(diagnostics.family.eq('primary20')&diagnostics.outcome.eq('log_sales'))|(diagnostics.family.eq('time40')&diagnostics.window.eq('post03'))]
display(focus[['family','industry','clusters','raw_temporally_changing_clusters','fe_variance_retained_pct','top5_information_share','inverse_herfindahl_information','partial_within_r2']])

### 시간창 공통 기울기의 정보 구성

공통 기울기는 전체 매출을 합산한 회귀도, 업종 계수를 단순 평균한 값도 아니다. 업종별 FE 잔차 노출 제곱합을 가중치로 한 기울기다. 같은 상권의 여러 업종을 합친 score로 표준오차를 계산한다.

In [ ]:
common_info=[]; common_weights=[]
for window,(blocks,common,heterogeneous) in models11.items():
    chunks=[]; total_ss=sum(float(b['x']@b['x']) for b in blocks)
    for i,b in enumerate(blocks):
        weight=float(b['x']@b['x'])/total_ss
        common_weights.append(dict(window=window,industry_id=b['industry_id'],industry=INDUSTRIES[b['industry_id']],information_weight=weight,industry_coefficient=float(heterogeneous['beta'][i])))
        chunks.append(pd.DataFrame({'area_id':b['frame'].area_id.to_numpy(),'industry_id':b['industry_id'],
          'xraw':b['frame'].x.to_numpy(),'xr':b['x'],'score':b['x']*(b['y']-b['x']*common['beta'][0])}))
    joined=pd.concat(chunks,ignore_index=True)
    info=joined.assign(info=joined.xr**2).groupby('area_id')['info'].sum()
    shares=(info/info.sum()).sort_values(ascending=False)
    score=joined.groupby('area_id').score.sum()
    recreated=float(np.sqrt(common['correction']*(score**2).sum())/total_ss)
    assert abs(recreated-np.sqrt(common['cov'][0,0]))<1e-10
    weights=[r for r in common_weights if r['window']==window]
    assert abs(sum(r['information_weight']*r['industry_coefficient'] for r in weights)-common['beta'][0])<1e-10
    centered_ss=sum(float(np.sum((b['frame'].x-b['frame'].x.mean())**2)) for b in blocks)
    common_info.append(dict(window=window,clusters=common['clusters'],timebin_rows=common['nobs'],base_area_industry_quarters=common['nobs']//6,
      unique_area_quarters=int(pd.concat([b['frame'][['area_id','quarter']] for b in blocks]).drop_duplicates().shape[0]),
      residual_x_ss=total_ss,fe_variance_retained_pct=100*total_ss/centered_ss,
      top5_information_share=float(shares.head(5).sum()),inverse_herfindahl_information=float(1/shares.pow(2).sum()),
      cluster_se_recreated=recreated))
common_info=pd.DataFrame(common_info); common_weights=pd.DataFrame(common_weights)
display(common_info)
display(common_weights.query("window == 'post03'").drop(columns='window'))

## 4. 계수·구간을 같은 노출 비교로 환산

`(1+노출)`가 2배이면 `Δx=ln(2)`이다. 노출 9→19와 같으며, 노출 자체의 2배와 다르다. 총매출·시간대 매출은 `100×[exp(βΔx)−1]`로 환산한 **(매출+1)의 조건부 기하평균 상대차이(%)**이다. 산술평균 매출 증가율이나 인과효과가 아니다. 야간 비중은 `βΔx` 퍼센트포인트다. 시간창 모형의 환산은 분기 공통 수준이 제거된 상대 패턴의 조건부 비교다.

표본에서 관측된 인접 분기 간 `|Δlog(1+노출)|` 중 양수 값의 중앙값도 보조 비교로 제시한다. 11은 같은 상권·업종·시간대끼리 비교한다. 이 값은 실제 분기 변화의 기술 요약이며, 해당 변화가 FE 제거 후에도 그대로 남는다거나 외생적이라는 뜻은 아니다. 2배 환산과 10→20 예시는 표본 밖 외삽 가능성이 있어 관측 범위 진단과 함께 읽는다.

In [ ]:
keys=['family','outcome','window','industry_id']
cols=['median_nonzero_adjacent_abs_delta_logx','clusters_spanning_log2_same_bin','adjacent_quarter_pairs','nonzero_adjacent_quarter_pairs','adjacent_quarter_pairs_ge_log2']
effects=effects.merge(diagnostics[keys+cols],on=keys,how='left',validate='one_to_one')
contrast_rows=[]
for row in effects.itertuples():
    contrasts=[('one_plus_exposure_doubled',np.log(2))]
    if row.family=='primary20': contrasts.append(('registered_IDs_10_to_20',np.log(21/11)))
    if np.isfinite(row.median_nonzero_adjacent_abs_delta_logx):
        contrasts.append(('median_nonzero_adjacent_quarter_change',row.median_nonzero_adjacent_abs_delta_logx))
    for name,delta in contrasts:
        vals=np.array([row.coefficient,row.ci_low,row.ci_high])*delta
        if row.outcome!='night_share_pct':
            vals=100*np.expm1(vals); unit='(sales+1) conditional geometric-mean relative difference (%)'
        else: unit='night share percentage points'
        contrast_rows.append(dict(family=row.family,outcome=row.outcome,window=row.window,industry_id=row.industry_id,industry=row.industry,
          contrast=name,delta_log_exposure=delta,estimate=vals[0],ci_low=vals[1],ci_high=vals[2],unit=unit,
          p_value=row.p_value,p_adjusted=row.p_adjusted))
contrasts=pd.DataFrame(contrast_rows)
display(contrasts.query("family == 'primary20' and contrast == 'one_plus_exposure_doubled'")
        [['outcome','industry','estimate','ci_low','ci_high','p_value','p_adjusted']])
display(contrasts.query("family == 'time_common4' and contrast == 'one_plus_exposure_doubled'")
        [['window','estimate','ci_low','ci_high','p_value','p_adjusted']])

### 보정 전후 결과는 각각 보고한다

보정 전 유의항목이 있는지, 그리고 보정 때문에만 결론이 바뀐 것인지 구분한다. BH는 선택된 20개/40개 가설군의 발견 오류 비율을 통제하기 위한 절차다. 보정 자체가 계수나 표준오차를 바꾸지는 않는다.

In [ ]:
multiplicity=effects.groupby('family').agg(tests=('p_value','size'),raw_significant=('p_value',lambda s:int(s.lt(.05).sum())),
    adjusted_significant=('p_adjusted',lambda s:int(s.lt(.05).sum())),minimum_raw_p=('p_value','min'),minimum_adjusted_p=('p_adjusted','min')).reset_index()
display(multiplicity)
display(effects.loc[effects.p_value.lt(.05),['family','outcome','window','industry','coefficient','standard_error','ci_low','ci_high','p_value','p_adjusted']])
display(effects.loc[effects.family.eq('time40'),['window','industry','coefficient','standard_error','ci_low','ci_high','p_value','p_adjusted']])

## 5. 고정 설계의 검출 규모 참고치

`[t(0.975,G−1)+z(0.80)]×SE`를 5% 양측 검정·약 80% 검출확률에 대응하는 **계수 규모의 근사치**로 제시한다. 현재 추정된 군집 SE를 미래에도 일정하다고 고정하고 주된 한쪽 꼬리를 근사한 값이다. 다중검정 전이며, BH/Holm 보정·측정오차·공간상관·대안 설계의 불확실성을 반영하지 않는다. 실제 효과가 이 정도라는 뜻이나 사후검정력이 아니다. 자료로부터 얻은 SE를 쓰는 조건부 기술 진단이므로 엄밀한 사전 표본 설계의 대체물도 아니다.

증가/감소 방향은 로그 역변환 때문에 대칭 %가 아니다. 환산 값은 모두 `(1+노출)` 2배 비교 기준이다.

In [ ]:
detect=effects[['family','outcome','window','industry_id','industry','standard_error','df']].copy()
detect['coefficient_magnitude_80_approx']=(stats.t.ppf(.975,detect.df)+stats.norm.ppf(.8))*detect.standard_error
delta=detect.coefficient_magnitude_80_approx*np.log(2)
night=detect.outcome.eq('night_share_pct')
detect['doubling_positive_80_approx']=np.where(night,delta,100*np.expm1(delta))
detect['doubling_negative_80_approx']=np.where(night,-delta,100*np.expm1(-delta))
detect['unit']=np.where(night,'percentage points','conditional geometric-mean relative difference (%)')
detect['multiplicity_adjusted']=False
display(detect.loc[detect.family.eq('primary20'),['outcome','industry','coefficient_magnitude_80_approx','doubling_negative_80_approx','doubling_positive_80_approx']])
display(detect.loc[detect.family.eq('time_common4'),['window','coefficient_magnitude_80_approx','doubling_negative_80_approx','doubling_positive_80_approx']])

## 6. 변동 소실과 집중도의 시각적 비교

세로축 업종은 코드 순서를 유지한다. 두 모형의 원시 x는 다른 노출 지표이므로 잔차 SD를 직접 같은 단위의 노출량으로 비교하지 않는다. 아래에서는 각 모형 안의 변동 보존율과 정보 집중도를 비교한다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
fonts={f.name for f in fm.fontManager.ttflist}
plt.rcParams['font.family']=next((x for x in ['Malgun Gothic','NanumGothic','AppleGothic'] if x in fonts),'DejaVu Sans')
plt.rcParams.update({'axes.unicode_minus':False,'font.size':10})
a=diagnostics.query("family == 'primary20' and outcome == 'log_sales'").set_index('industry_id').reindex(INDUSTRIES)
b=diagnostics.query("family == 'time40' and window == 'post03'").set_index('industry_id').reindex(INDUSTRIES)
y=np.arange(len(INDUSTRIES)); fig,axes=plt.subplots(1,2,figsize=(13,6.5),sharey=True)
for ax,column,factor,title in zip(axes,['fe_variance_retained_pct','top5_information_share'],[1,100],['고정효과 제거 후 원시 x 변동 보존율','상위 5개 상권의 잔차 x² 정보 비중']):
    ax.barh(y-.18,a[column]*factor,height=.34,label='분기 총매출 모형',color='#266987')
    ax.barh(y+.18,b[column]*factor,height=.34,label='종료 후 0~3h 상대 패턴 모형',color='#CB8350')
    ax.set_yticks(y,list(INDUSTRIES.values())); ax.set_ylim(9.5,-.5); ax.set_xlim(0,100)
    ax.set_xlabel('%'); ax.set_title(title); ax.grid(axis='x',alpha=.2)
axes[1].legend(loc='lower right',fontsize=8)
fig.suptitle('행 수와 실제 식별 정보는 다르다 · 고정된 주모형 진단')
fig.tight_layout()
fig.savefig(OUT/'inference_variation_concentration.png',dpi=160,bbox_inches='tight')
plt.show(); plt.close(fig)

## 7. 근본 원인에 대해 말할 수 있는 범위

1. **질문과 추정 대상의 차이:** 공연의 실제 개최·종료·관람객을 관측한 처치 효과가 아니다. 특히 시간창 3중 FE 모형은 총매출 증가를 제거한 뒤 상대 시간대 패턴을 추정한다. 총매출이 모든 시간대에 비슷하게 움직이는 경우는 이 모형의 검정 대상이 아니다.
2. **실제로 사용되는 노출 변동의 제한:** 지속적인 공연장 밀도 차이나 공통 분기 변화는 FE에 흡수된다. 잔차 변동 비중과 정보 집중도는 그 결과를 수량화한다. 제거된 변동을 되살려 유의성을 만드는 것은 원래 질문을 바꾸며 교란 위험도 바꾼다.
3. **검정 정밀도:** CI와 고정 SE 검출규모로 판별 가능한 효과 크기를 보여 준다. 넓은 구간은 양·음의 관계가 모두 자료와 양립함을 뜻한다. 작은 점추정치만으로 경제적 중요성이나 효과 부재를 단정할 수 없다.
4. **다중검정의 역할:** 원시 p값과 보정값을 함께 보여야 한다. 40개 시간창 업종검정은 보정 전에도 모두 비유의하므로 이 결과를 다중검정 보정만의 결과로 설명할 수 없다.
5. **측정·선택 문제:** ID와 실제 회차·관객의 차이, 예정 일정 반복 가정, 분기×넓은 시간구간, 선택적 매출 미제공은 실제 신호를 흐리거나 추정 대상을 바꿀 수 있다. 그러나 현재 자료만으로 편향 방향·크기나 각각이 비유의성을 초래한 비율은 식별되지 않는다. 단순히 측정오차 때문에 0으로 축소되었다고 단정하지 않는다.

따라서 “실제 효과 없음”, “표본만 늘리면 유의”, “모형이 너무 엄격해 유의성이 사라짐” 중 하나를 확정 원인으로 고르는 것은 근거를 넘는다. 현재 자료로는 어떤 관계를 얼마나 정밀하게 추정했는지까지 설명한다.

In [ ]:
p=a; t=b
primary_time=contrasts.query("family == 'time_common4' and window == 'post03' and contrast == 'one_plus_exposure_doubled'").iloc[0]
lines=[f"20개 기본 검정 중 보정 전 유의 {int(effects.query("family == 'primary20'").p_value.lt(.05).sum())}개, BH 후 0개; 시간창 40개는 보정 전후 모두 0개.",
 f"기본 총매출 모형: 업종별 FE 후 원시 log 노출 변동 {p.fe_variance_retained_pct.min():.2f}~{p.fe_variance_retained_pct.max():.2f}% 보존, 상위 5개 정보 비중 {100*p.top5_information_share.min():.1f}~{100*p.top5_information_share.max():.1f}%.",
 f"종료 후 0~3h 업종 모형: FE 후 변동 {t.fe_variance_retained_pct.min():.2f}~{t.fe_variance_retained_pct.max():.2f}% 보존, 상위 5개 정보 비중 {100*t.top5_information_share.min():.1f}~{100*t.top5_information_share.max():.1f}%.",
 f"공통 시간창 (1+노출) 2배 비교: {primary_time.estimate:+.2f}%, 개별 95% CI [{primary_time.ci_low:+.2f}%, {primary_time.ci_high:+.2f}%]. 총매출 성장률이나 관객 소비의 인과효과가 아니다."]
display(Markdown('\n\n'.join(lines)))
outputs={'inference_reproduction.csv':reproduction,'inference_effects.csv':effects,'inference_identifying_variation.csv':diagnostics,
 'inference_cluster_information.csv':cluster_info,'inference_common_information.csv':common_info,'inference_common_industry_weights.csv':common_weights,
 'inference_contrasts.csv':contrasts,'inference_multiplicity.csv':multiplicity,'inference_detection_scale.csv':detect}
for name,frame in outputs.items():
    assert name.startswith('inference_')
    frame.to_csv(OUT/name,index=False,encoding='utf-8-sig')
assert all(sha(project_path(path))==value for path,value in protected.items()),'기존 입력 또는 결과가 변경되었습니다.'
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'scope':'fixed original main models; precision and identifying variation only',
 'network_requests':0,'specification_search':False,'observed_posthoc_power_calculated':False,
 'reproduced_coefficients':len(reproduction),'maximum_reproduction_error':float(reproduction.max_absolute_error.max()),
 'protected_unchanged':True,'protected_sha256':protected,'multiplicity':multiplicity.to_dict('records'),
 'headline_diagnostics':lines,'detection_scale_definition':'(t_0.975_df + z_0.80)*current_cluster_SE; approximate, unadjusted, fixed design',
 'causal_conclusion':False,'python':platform.python_version(),
 'outputs_sha256':{name:sha(OUT/name) for name in [*outputs,'inference_variation_concentration.png']}}
(OUT/'inference_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
print('진단 저장 완료 / 원본·기존 분석 결과 SHA-256 불변 확인')